# 📖 Notebook 2: Windowed Aggregations

Raw time-series data at 30-second resolution is noisy. To make it useful we need to **bucket** it into time windows and compute aggregates like averages, maximums, and percentiles.

## Learning Objectives

By the end of this notebook you will understand:
- How to use TimescaleDB's `time_bucket()` to group data into fixed intervals
- Running aggregations: `AVG`, `MAX`, `MIN`, `COUNT` over time windows
- PostgreSQL window functions for moving averages and ranking
- Comparing multiple hosts side-by-side with grouped time buckets

## 🛠️ Setup

Make sure the containers are running:

```bash
cd deep-dives/time-series-databases
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "tsdb_demo",
    "user": "demo",
    "password": "demo"
}

def run_query(sql, params=None):
    with psycopg2.connect(**DB_CONFIG) as conn:
        return pd.read_sql_query(sql, conn, params=params)

print("✅ Connected!")

---
## 1 — `time_bucket()`: The Core Tool

Think of `time_bucket()` like rounding timestamps down to the nearest interval.  
If the interval is `5 minutes`, then `14:03:22` becomes `14:00:00` and `14:07:45` becomes `14:05:00`.

This lets us group all data points that fall within the same 5-minute window.

```sql
time_bucket('5 minutes', time)  -- rounds time down to 5-minute boundaries
```

Let's use it to compute 5-minute average CPU for one host.

In [ ]:
df = run_query("""
    SELECT
        time_bucket('5 minutes', time) AS bucket,
        avg(value)                     AS avg_cpu,
        min(value)                     AS min_cpu,
        max(value)                     AS max_cpu,
        count(*)                       AS samples
    FROM metrics
    WHERE host = 'server-1'
      AND metric_name = 'cpu_usage'
      AND time > now() - interval '6 hours'
    GROUP BY bucket
    ORDER BY bucket
""")

print(f"5-minute buckets for the last 6 hours: {len(df)} rows")
df.head(10)

In [ ]:
# Visualize: raw data vs. 5-minute average
raw = run_query("""
    SELECT time, value
    FROM metrics
    WHERE host = 'server-1'
      AND metric_name = 'cpu_usage'
      AND time > now() - interval '6 hours'
    ORDER BY time
""")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(raw['time'], raw['value'], alpha=0.3, linewidth=0.5, label='Raw (30s)')
ax.plot(df['bucket'], df['avg_cpu'], color='red', linewidth=1.5, label='5-min avg')
ax.fill_between(df['bucket'], df['min_cpu'], df['max_cpu'], alpha=0.1, color='red', label='5-min min/max')
ax.set_title('CPU Usage — server-1 (raw vs. 5-min aggregation)')
ax.set_xlabel('Time')
ax.set_ylabel('CPU %')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Notice how the red line smooths out the noise while the min/max band shows the full range.  
This is the same principle behind **downsampling** — we trade resolution for clarity and storage savings.

---
## 2 — Different Bucket Sizes

The right bucket size depends on what you're looking at:
- **1 minute** — debugging a recent spike
- **15 minutes** — last-day dashboard
- **1 hour** — weekly trend
- **1 day** — monthly capacity planning

Let's compare them.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=False)
buckets = ['1 minute', '15 minutes', '1 hour', '1 day']
lookbacks = ['2 hours', '24 hours', '7 days', '7 days']

for ax, bucket, lookback in zip(axes.flat, buckets, lookbacks):
    data = run_query(f"""
        SELECT
            time_bucket('{bucket}', time) AS bucket,
            avg(value) AS avg_cpu
        FROM metrics
        WHERE host = 'server-1'
          AND metric_name = 'cpu_usage'
          AND time > now() - interval '{lookback}'
        GROUP BY bucket
        ORDER BY bucket
    """)
    ax.plot(data['bucket'], data['avg_cpu'], linewidth=1)
    ax.set_title(f'Bucket: {bucket}  (lookback: {lookback})')
    ax.set_ylabel('CPU %')
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Effect of bucket size on CPU time series', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3 — Comparing Multiple Hosts

A common dashboard pattern: show the same metric for several hosts side-by-side.

In [ ]:
multi = run_query("""
    SELECT
        time_bucket('15 minutes', time) AS bucket,
        host,
        avg(value) AS avg_cpu
    FROM metrics
    WHERE metric_name = 'cpu_usage'
      AND host IN ('server-1', 'server-3', 'server-7')
      AND time > now() - interval '24 hours'
    GROUP BY bucket, host
    ORDER BY bucket
""")

fig, ax = plt.subplots(figsize=(14, 5))
for host, group in multi.groupby('host'):
    ax.plot(group['bucket'], group['avg_cpu'], label=host, linewidth=1)
ax.set_title('15-min avg CPU — multiple hosts')
ax.set_ylabel('CPU %')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4 — Moving Averages with Window Functions

A **moving average** smooths a time series by averaging over a sliding window of recent values.  
PostgreSQL's `AVG() OVER (...)` window function makes this easy.

We'll compute a 30-minute moving average (6 buckets × 5 min each).

In [ ]:
ma = run_query("""
    WITH buckets AS (
        SELECT
            time_bucket('5 minutes', time) AS bucket,
            avg(value) AS avg_cpu
        FROM metrics
        WHERE host = 'server-1'
          AND metric_name = 'cpu_usage'
          AND time > now() - interval '12 hours'
        GROUP BY bucket
        ORDER BY bucket
    )
    SELECT
        bucket,
        avg_cpu,
        avg(avg_cpu) OVER (
            ORDER BY bucket
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ) AS moving_avg_30m
    FROM buckets
""")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(ma['bucket'], ma['avg_cpu'], alpha=0.4, linewidth=0.8, label='5-min avg')
ax.plot(ma['bucket'], ma['moving_avg_30m'], color='red', linewidth=2, label='30-min moving avg')
ax.set_title('Moving Average — server-1 CPU')
ax.set_ylabel('CPU %')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5 — Finding Anomalies: Percentiles and Spikes

Averages hide outliers. Let's find time windows where a metric spiked above the 95th percentile.

In [ ]:
spikes = run_query("""
    WITH bucket_stats AS (
        SELECT
            time_bucket('5 minutes', time) AS bucket,
            host,
            avg(value) AS avg_val,
            max(value) AS max_val
        FROM metrics
        WHERE metric_name = 'disk_io'
          AND time > now() - interval '24 hours'
        GROUP BY bucket, host
    ),
    thresholds AS (
        SELECT percentile_cont(0.95) WITHIN GROUP (ORDER BY avg_val) AS p95
        FROM bucket_stats
    )
    SELECT bs.bucket, bs.host, bs.avg_val, bs.max_val
    FROM bucket_stats bs, thresholds t
    WHERE bs.avg_val > t.p95
    ORDER BY bs.avg_val DESC
    LIMIT 15
""")

print("Top disk I/O spikes (above 95th percentile):")
spikes

---
## 6 — Region-Level Rollup

In real monitoring you often want a **region-level** view: "What is the average CPU across all servers in us-west?"

In [ ]:
region = run_query("""
    SELECT
        time_bucket('1 hour', time) AS hour,
        region,
        avg(value) AS avg_cpu,
        max(value) AS max_cpu,
        count(DISTINCT host) AS hosts
    FROM metrics
    WHERE metric_name = 'cpu_usage'
      AND time > now() - interval '3 days'
    GROUP BY hour, region
    ORDER BY hour
""")

fig, ax = plt.subplots(figsize=(14, 5))
for r, group in region.groupby('region'):
    ax.plot(group['hour'], group['avg_cpu'], label=f"{r} (avg)", linewidth=1.2)
    ax.fill_between(group['hour'], group['avg_cpu'], group['max_cpu'], alpha=0.1)
ax.set_title('Hourly CPU by Region (last 3 days)')
ax.set_ylabel('CPU %')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 🧠 Key Takeaways

1. **`time_bucket()`** is the Swiss Army knife — it rounds timestamps into fixed windows for GROUP BY.
2. Choose bucket sizes based on your use case: small for debugging, large for trends.
3. **Window functions** (`AVG() OVER`, `LAG()`, `RANK()`) let you compute moving averages and detect changes.
4. **Percentiles** help spot anomalies that averages would hide.
5. Roll up by tags (region, service) to get higher-level views without losing the ability to drill down.

**Next notebook →** We'll set up retention policies to automatically delete old data and continuous aggregates to pre-compute downsampled rollups.